# Assignment 3b: Advanced Gradio RAG Frontend
## Day 6 Session 2 - Building Configurable RAG Applications

In this assignment, you'll extend your basic RAG interface with advanced configuration options to create a professional, feature-rich RAG application.

**New Features to Add:**
- Model selection dropdown (gpt-4o, gpt-4o-mini)
- Temperature slider (0 to 1 with 0.1 intervals)
- Chunk size configuration
- Chunk overlap configuration  
- Similarity top-k slider
- Node postprocessor multiselect
- Similarity cutoff slider
- Response synthesizer multiselect

**Learning Objectives:**
- Advanced Gradio components and interactions
- Dynamic RAG configuration
- Professional UI design patterns
- Parameter validation and handling
- Building production-ready AI applications

**Prerequisites:**
- Completed Assignment 3a (Basic Gradio RAG)
- Understanding of RAG parameters and their effects


## 📚 Part 1: Setup and Imports

Import all necessary libraries including advanced RAG components for configuration options.

**Note:** This assignment uses OpenRouter for LLM access (not OpenAI). Make sure you have your `OPENROUTER_API_KEY` environment variable set.


In [1]:
# Install required packages
!pip install gradio llama-index llama-index-llms-openrouter llama-index-embeddings-openai llama-index-vector-stores-lancedb lancedb pypdf llama-index-readers-file -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.9/394.9 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 11.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [3]:
import warnings
warnings.filterwarnings("ignore")

import os
import gradio as gr
from pathlib import Path

from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
    Settings,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.response_synthesizers import get_response_synthesizer
from llama_index.core.response_synthesizers.type import ResponseMode
from llama_index.vector_stores.lancedb import LanceDBVectorStore
from llama_index.llms.openrouter import OpenRouter
from llama_index.embeddings.openai import OpenAIEmbedding

# Set API key
from google.colab import userdata
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY").strip()

print("✅ All libraries imported successfully!")
print(f"   Gradio version: {gr.__version__}")

SecretNotFoundError: Secret OPENROUTER_API_KEY does not exist.

In [5]:
# Set API key
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY").strip()

print("✅ All libraries imported successfully!")
print(f"   Gradio version: {gr.__version__}")

SecretNotFoundError: Secret OPENAI_API_KEY does not exist.

In [7]:
from google.colab import userdata
import os
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY").strip()
print("Key loaded! Length:", len(os.environ["OPENROUTER_API_KEY"]))

SecretNotFoundError: Secret OPENROUTER_API_KEY does not exist.

In [9]:
from google.colab import userdata
import os
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY").strip()
print("Key loaded! Length:", len(os.environ["OPENROUTER_API_KEY"]))

Key loaded! Length: 73


In [ ]:
# Import all required libraries
import gradio as gr
import os
from pathlib import Path
from typing import Dict, List, Optional, Any

# LlamaIndex core components
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext, Settings
from llama_index.vector_stores.lancedb import LanceDBVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openrouter import OpenRouter

# Advanced RAG components
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.response_synthesizers import TreeSummarize, Refine, CompactAndRefine
from llama_index.core.retrievers import VectorIndexRetriever

print("✅ All libraries imported successfully!")


/Users/ishandutta/miniconda3/envs/accelerator/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All libraries imported successfully!


## 🤖 Part 2: Advanced RAG Backend Class

Create an advanced RAG backend that supports dynamic configuration of all parameters.


In [11]:
from llama_index.llms.openrouter import OpenRouter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.response_synthesizers import TreeSummarize, CompactAndRefine
from llama_index.core.response_synthesizers import Refine
from typing import Dict, List, Any

class AdvancedRAGBackend:

    def __init__(self):
        self.index = None
        self.available_models = ["openai/gpt-4o", "openai/gpt-4o-mini"]
        self.available_postprocessors = ["SimilarityPostprocessor"]
        self.available_synthesizers = ["TreeSummarize", "Refine", "CompactAndRefine", "Default"]
        self.update_settings()

    def update_settings(self, model="openai/gpt-4o-mini", temperature=0.1, chunk_size=512, chunk_overlap=50):
        api_key = os.getenv("OPENROUTER_API_KEY")
        if api_key:
            Settings.llm = OpenRouter(
                api_key=api_key,
                model=model,
                temperature=temperature
            )
        Settings.embed_model = OpenAIEmbedding(
            model="text-embedding-3-small",
            api_key=api_key,
            api_base="https://openrouter.ai/api/v1"
        )
        Settings.chunk_size = chunk_size
        Settings.chunk_overlap = chunk_overlap

    def initialize_database(self, data_folder="/content/ai-accelerator-C5/Day_6/session_2/data/"):
        if not Path(data_folder).exists():
            return f"Data folder '{data_folder}' not found!"
        try:
            from llama_index.core.schema import Document
            vector_store = LanceDBVectorStore(
                uri="./advanced_rag_vectordb",
                table_name="documents",
                mode="overwrite"
            )
            raw_docs = SimpleDirectoryReader(
                input_dir=data_folder,
                recursive=True,
                exclude=["*.mp3", "*.mp4", "*.wav", "*.m4a"],
                errors="ignore"
            ).load_data()

            documents = [
                Document(
                    text=doc.text.encode("ascii", errors="ignore").decode("ascii"),
                    metadata=doc.metadata
                )
                for doc in raw_docs
            ]

            storage_context = StorageContext.from_defaults(vector_store=vector_store)
            self.index = VectorStoreIndex.from_documents(
                documents,
                storage_context=storage_context,
                show_progress=True
            )
            return f"✅ Database initialized with {len(documents)} documents!"
        except Exception as e:
            return f"❌ Error: {str(e)}"

    def get_postprocessor(self, name, similarity_cutoff):
        if name == "SimilarityPostprocessor":
            return SimilarityPostprocessor(similarity_cutoff=similarity_cutoff)
        return None

    def get_synthesizer(self, name):
        if name == "TreeSummarize":
            return TreeSummarize()
        elif name == "Refine":
            return Refine()
        elif name == "CompactAndRefine":
            return CompactAndRefine()
        return None

    def advanced_query(self, question, model, temperature, chunk_size, chunk_overlap,
                      similarity_top_k, postprocessor_names, similarity_cutoff, synthesizer_name):
        if self.index is None:
            return {"response": "Please initialize the database first!", "sources": [], "config": {}}
        if not question or not question.strip():
            return {"response": "Please enter a question!", "sources": [], "config": {}}
        try:
            self.update_settings(model, temperature, chunk_size, chunk_overlap)
            postprocessors = [
                self.get_postprocessor(name, similarity_cutoff)
                for name in postprocessor_names
                if self.get_postprocessor(name, similarity_cutoff) is not None
            ]
            synthesizer = self.get_synthesizer(synthesizer_name)
            kwargs = {"similarity_top_k": similarity_top_k}
            if postprocessors:
                kwargs["node_postprocessors"] = postprocessors
            if synthesizer:
                kwargs["response_synthesizer"] = synthesizer
            query_engine = self.index.as_query_engine(**kwargs)
            response = query_engine.query(question)
            sources = []
            if hasattr(response, "source_nodes"):
                for node in response.source_nodes:
                    sources.append({
                        "text": node.text[:200] + "...",
                        "score": getattr(node, "score", 0.0),
                        "source": node.node.metadata.get("file_name", "Unknown")
                    })
            return {
                "response": str(response),
                "sources": sources,
                "config": {
                    "model": model,
                    "temperature": temperature,
                    "chunk_size": chunk_size,
                    "similarity_top_k": similarity_top_k,
                    "synthesizer": synthesizer_name
                }
            }
        except Exception as e:
            return {"response": f"Error: {str(e)}", "sources": [], "config": {}}

rag_backend = AdvancedRAGBackend()
print("✅ Advanced RAG Backend initialized!")

✅ Advanced RAG Backend initialized!


## 🎨 Part 3: Advanced Gradio Interface

Create a sophisticated Gradio interface with all the configuration options specified:
1. Database initialization button
2. Search query input and button  
3. Model selection dropdown
4. Temperature slider
5. Chunk size and overlap inputs
6. Similarity top-k slider
7. Node postprocessor multiselect
8. Similarity cutoff slider
9. Response synthesizer multiselect


In [12]:
def create_advanced_rag_interface():

    def initialize_db():
        return rag_backend.initialize_database()

    def handle_advanced_query(question, model, temperature, chunk_size, chunk_overlap,
                             similarity_top_k, postprocessors, similarity_cutoff, synthesizer):
        result = rag_backend.advanced_query(
            question, model, temperature, chunk_size, chunk_overlap,
            similarity_top_k, postprocessors, similarity_cutoff, synthesizer
        )
        config_text = f"""Current Configuration:
- Model: {result['config'].get('model', 'N/A')}
- Temperature: {result['config'].get('temperature', 'N/A')}
- Chunk Size: {result['config'].get('chunk_size', 'N/A')}
- Similarity Top-K: {result['config'].get('similarity_top_k', 'N/A')}
- Synthesizer: {result['config'].get('synthesizer', 'N/A')}"""
        return result["response"], config_text

    with gr.Blocks(title="Advanced RAG Assistant", theme=gr.themes.Soft()) as interface:

        gr.Markdown("# Advanced RAG Assistant")
        gr.Markdown("Configure every RAG parameter and ask questions about your documents.")

        with gr.Row():
            init_btn = gr.Button("Initialize Database", variant="primary")
            status_output = gr.Textbox(label="Database Status", interactive=False)

        init_btn.click(initialize_db, outputs=[status_output])

        with gr.Row():

            with gr.Column(scale=1):
                gr.Markdown("### Configuration")

                model_dropdown = gr.Dropdown(
                    label="Model",
                    choices=["openai/gpt-4o", "openai/gpt-4o-mini"],
                    value="openai/gpt-4o-mini"
                )

                temperature_slider = gr.Slider(
                    label="Temperature",
                    minimum=0.0,
                    maximum=1.0,
                    step=0.1,
                    value=0.1
                )

                chunk_size_input = gr.Number(
                    label="Chunk Size",
                    value=512
                )

                chunk_overlap_input = gr.Number(
                    label="Chunk Overlap",
                    value=50
                )

                similarity_topk_slider = gr.Slider(
                    label="Similarity Top-K",
                    minimum=1,
                    maximum=20,
                    step=1,
                    value=5
                )

                postprocessor_checkbox = gr.CheckboxGroup(
                    label="Node Postprocessors",
                    choices=["SimilarityPostprocessor"],
                    value=[]
                )

                similarity_cutoff_slider = gr.Slider(
                    label="Similarity Cutoff",
                    minimum=0.0,
                    maximum=1.0,
                    step=0.1,
                    value=0.3
                )

                synthesizer_dropdown = gr.Dropdown(
                    label="Response Synthesizer",
                    choices=["TreeSummarize", "Refine", "CompactAndRefine", "Default"],
                    value="Default"
                )

            with gr.Column(scale=2):
                gr.Markdown("### Query Interface")

                query_input = gr.Textbox(
                    label="Ask a question",
                    placeholder="e.g. What are AI agents?",
                    lines=3
                )

                submit_btn = gr.Button("Ask", variant="primary")

                response_output = gr.Textbox(
                    label="Response",
                    lines=12,
                    interactive=False
                )

                config_display = gr.Textbox(
                    label="Configuration Used",
                    lines=8,
                    interactive=False
                )

        submit_btn.click(
            handle_advanced_query,
            inputs=[
                query_input, model_dropdown, temperature_slider,
                chunk_size_input, chunk_overlap_input, similarity_topk_slider,
                postprocessor_checkbox, similarity_cutoff_slider, synthesizer_dropdown
            ],
            outputs=[response_output, config_display]
        )

        query_input.submit(
            handle_advanced_query,
            inputs=[
                query_input, model_dropdown, temperature_slider,
                chunk_size_input, chunk_overlap_input, similarity_topk_slider,
                postprocessor_checkbox, similarity_cutoff_slider, synthesizer_dropdown
            ],
            outputs=[response_output, config_display]
        )

    return interface

advanced_interface = create_advanced_rag_interface()
print("✅ Advanced RAG interface created successfully!")

✅ Advanced RAG interface created successfully!


## 🚀 Part 4: Launch Your Advanced Application

Launch your advanced Gradio application and test all the configuration options!


In [13]:
print("🎉 Launching your Advanced RAG Assistant...")
print("🔗 Your application will open in a new browser tab!")
print("")
print("⚠️  Make sure your OPENROUTER_API_KEY environment variable is set!")
print("")
print("📋 Testing Instructions:")
print("1. Click 'Initialize Vector Database' button first")
print("2. Wait for success message")
print("3. Configure your RAG parameters:")
print("   - Choose model (gpt-4o, gpt-4o-mini)")
print("   - Adjust temperature (0.0 = deterministic, 1.0 = creative)")
print("   - Set chunk size and overlap")
print("   - Choose similarity top-k")
print("   - Select postprocessors and synthesizer")
print("4. Enter a question and click 'Ask Question'")
print("5. Review both the response and configuration used")
print("")
print("🧪 Experiments to try:")
print("- Compare different models with the same question")
print("- Test temperature effects (0.1 vs 0.9)")
print("- Try different chunk sizes (256 vs 1024)")
print("- Compare synthesizers (TreeSummarize vs Refine)")
print("- Adjust similarity cutoff to filter results")

# Your code here:
advanced_interface.launch()

🎉 Launching your Advanced RAG Assistant...
🔗 Your application will open in a new browser tab!

⚠️  Make sure your OPENROUTER_API_KEY environment variable is set!

📋 Testing Instructions:
1. Click 'Initialize Vector Database' button first
2. Wait for success message
3. Configure your RAG parameters:
   - Choose model (gpt-4o, gpt-4o-mini)
   - Adjust temperature (0.0 = deterministic, 1.0 = creative)
   - Set chunk size and overlap
   - Choose similarity top-k
   - Select postprocessors and synthesizer
4. Enter a question and click 'Ask Question'
5. Review both the response and configuration used

🧪 Experiments to try:
- Compare different models with the same question
- Test temperature effects (0.1 vs 0.9)
- Try different chunk sizes (256 vs 1024)
- Compare synthesizers (TreeSummarize vs Refine)
- Adjust similarity cutoff to filter results
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
!git clone https://github.com/eng-accelerator/ai-accelerator-C5.git
import os
print(os.path.exists("/content/ai-accelerator-C5/Day_6/session_2/data/"))

Cloning into 'ai-accelerator-C5'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 150 (delta 14), reused 4 (delta 4), pack-reused 127 (from 2)
Receiving objects: 100% (150/150), 26.04 MiB | 39.50 MiB/s, done.
Resolving deltas: 100% (25/25), done.
True


## 💡 Understanding the Configuration Options

### Model Selection
- **gpt-4o**: Latest and most capable model, best quality responses
- **gpt-4o-mini**: Faster and cheaper while maintaining good quality

### Temperature (0.0 - 1.0)
- **0.0-0.3**: Deterministic, factual responses
- **0.4-0.7**: Balanced creativity and accuracy
- **0.8-1.0**: More creative and varied responses

### Chunk Size & Overlap
- **Chunk Size**: How much text to process at once (256-1024 typical)
- **Chunk Overlap**: Overlap between chunks to maintain context (10-100 typical)

### Similarity Top-K (1-20)
- **Lower values (3-5)**: More focused, faster responses
- **Higher values (8-15)**: More comprehensive, detailed responses

### Node Postprocessors
- **SimilarityPostprocessor**: Filters out low-relevance documents

### Similarity Cutoff (0.0-1.0)
- **0.1-0.3**: More permissive, includes potentially relevant docs
- **0.5-0.8**: More strict, only highly relevant docs

### Response Synthesizers
- **TreeSummarize**: Hierarchical summarization, good for complex topics
- **Refine**: Iterative refinement, builds detailed responses
- **CompactAndRefine**: Efficient version of Refine
- **Default**: Standard synthesis approach


## ✅ Assignment Completion Checklist

Before submitting, ensure you have:

- [ ] Set up your OPENROUTER_API_KEY environment variable
- [ ] Imported all necessary libraries including advanced RAG components
- [ ] Created AdvancedRAGBackend class with configurable parameters
- [ ] Implemented all required methods:
  - [ ] `update_settings()` - Updates LLM and chunking parameters
  - [ ] `initialize_database()` - Sets up vector database
  - [ ] `get_postprocessor()` - Returns selected postprocessor
  - [ ] `get_synthesizer()` - Returns selected synthesizer
  - [ ] `advanced_query()` - Handles queries with all configuration options
- [ ] Created advanced Gradio interface with all required components:
  - [ ] Initialize database button
  - [ ] Model selection dropdown (gpt-4o, gpt-4o-mini)
  - [ ] Temperature slider (0 to 1, step 0.1)
  - [ ] Chunk size input (default 512)
  - [ ] Chunk overlap input (default 50)
  - [ ] Similarity top-k slider (1 to 20, default 5)
  - [ ] Node postprocessor multiselect
  - [ ] Similarity cutoff slider (0.0 to 1.0, step 0.1, default 0.3)
  - [ ] Response synthesizer dropdown
  - [ ] Query input and submit button
  - [ ] Response output
  - [ ] Configuration display
- [ ] Connected all components to backend functions
- [ ] Successfully launched the application
- [ ] Tested different parameter combinations
- [ ] Verified all configuration options work correctly

## 🎊 Congratulations!

You've successfully built a professional, production-ready RAG application! You now have:

- **Advanced Parameter Control**: Full control over all RAG system parameters
- **Professional UI**: Clean, organized interface with proper layout
- **Real-time Configuration**: Ability to experiment with different settings
- **Production Patterns**: Understanding of how to build scalable AI applications

## 🚀 Next Steps & Extensions

**Potential Enhancements:**
1. **Authentication**: Add user login and session management
2. **Document Upload**: Allow users to upload their own documents
3. **Chat History**: Implement conversation memory
4. **Performance Monitoring**: Add response time and quality metrics
5. **A/B Testing**: Compare different configurations side-by-side
6. **Export Features**: Download responses and configurations
7. **Advanced Visualizations**: Show document similarity scores and retrieval paths

**Deployment Options:**
- **Local**: Run on your machine for development
- **Gradio Cloud**: Deploy with `interface.launch(share=True)`
- **Hugging Face Spaces**: Deploy to Hugging Face for public access
- **Docker**: Containerize for scalable deployment
- **Cloud Platforms**: Deploy to AWS, GCP, or Azure

You're now ready to build sophisticated AI-powered applications!
